# ChE629 Course Project

In [1]:
import torch
import numpy as np
import pandas as pd
import sklearn
import os
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
print("All libraries working perfectly")

All libraries working perfectly


In [3]:
# Local paths
root = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/"

davis_path = os.path.join(root, "EDTA-main/data/davis/SMILES_affinity.csv")
kiba_path  = os.path.join(root, "EDTA-main/data/kiba/SMILES_affinity.csv")

# Load datasets
davis_data = pd.read_csv(davis_path)
kiba_data = pd.read_csv(kiba_path)

print("DAVIS shape:", davis_data.shape)
print("KIBA shape:", kiba_data.shape)

DAVIS shape: (30056, 3)
KIBA shape: (118254, 3)


#### Input Features Processing

b) Drug Features

In [ ]:
max_atoms = 50
feature_dim = 24

print(f'Maximum atoms for features are taken into account (per molecule): {max_atoms}')

def pad_or_truncate(mat):
    num_atoms = mat.shape[0]

    # truncate
    if num_atoms > max_atoms:
        return mat[:max_atoms, :]

    # pad
    if num_atoms < max_atoms:
        pad_rows = max_atoms - num_atoms
        padding = np.zeros((pad_rows, feature_dim))
        return np.vstack([mat, padding])

    return mat


def load_drug_tensor(pkl_path):
    
    daf = pd.read_pickle(pkl_path)
    daf_fixed = {k: pad_or_truncate(v) for k, v in daf.items()}
    matrix_stack = np.stack(list(daf_fixed.values()))
    drug_tensor = torch.tensor(matrix_stack, dtype=torch.float32)

    return drug_tensor, list(daf_fixed.keys())


# KIBA
kiba_path  = os.path.join(root, "EDTA-main/data/kiba/Drug_Atomic_Feature24.pkl")
kiba_drug_tensor, kiba_drug_keys = load_drug_tensor(kiba_path)
print("KIBA tensor shape:", kiba_drug_tensor.shape)

# DAVIS
davis_path = os.path.join(root, "EDTA-main/data/davis/Drug_Atomic_Feature24.pkl")
davis_drug_tensor, davis_drug_keys = load_drug_tensor(davis_path)
print("DAVIS tensor shape:", davis_drug_tensor.shape)

Maximum atoms for features are taken into account (per molecule): 50
KIBA tensor shape: torch.Size([2111, 50, 24])
DAVIS tensor shape: torch.Size([68, 50, 24])


a) Drug SMILES

This character level enconding could be replaced with atom level encoding. Like convert C,l to Cl and B,r to Br respectively.

In [5]:
# Combine SMILES from both datasets
combined_smiles = pd.concat([davis_data['ligand'], kiba_data['ligand']])

# Build vocabulary
chars = sorted(set("".join(combined_smiles)))
char_to_int = {c: i+1 for i, c in enumerate(chars)}

print("Vocabulary size:", len(chars))
print("Characters:", chars)
SMILES_VOCAB_SIZE = len(chars) + 1

Vocabulary size: 32
Characters: ['#', '(', ')', '+', '-', '.', '/', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '@', 'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', '[', '\\', ']', 'l', 'r']


In [6]:
unknown_chars = set()
for smiles in kiba_drug_keys:
    for c in smiles:
        if c not in char_to_int:
            unknown_chars.add(c)
print("Unknown characters in KIBA drug keys:", unknown_chars)  # should be empty set

Unknown characters in KIBA drug keys: set()


In [ ]:
unknown_chars = set()
for smiles in davis_drug_keys:
    for c in smiles:
        if c not in char_to_int:
            unknown_chars.add(c)
print("Unknown characters in DAVIS drug keys:", unknown_chars)

Unknown characters in DAVIS drug keys: set()


In [ ]:
# Encoding function
def encode_smiles(smiles):
    return [char_to_int[c] for c in smiles]

# Set max length
max_len = 100
print("Using max SMILES length:", max_len)

# Pad / truncate
def pad_or_truncate_smiles(vec):
    if len(vec) > max_len:
        return vec[:max_len]
    return vec + [0]*(max_len - len(vec))

# DAVIS (Build SMILES tensor)
davis_vectors = [
    pad_or_truncate_smiles(encode_smiles(smiles)) 
    for smiles in davis_drug_keys
]
davis_smiles_tensor = torch.tensor(np.stack(davis_vectors), dtype=torch.long)
print("Davis tensor shape:", davis_smiles_tensor.shape)

# KIBA (Build SMILES tensor)
kiba_vectors = [
    pad_or_truncate_smiles(encode_smiles(smiles))
    for smiles in kiba_drug_keys
]
kiba_smiles_tensor = torch.tensor(np.stack(kiba_vectors), dtype=torch.long)
print("KIBA SMILES tensor shape:", kiba_smiles_tensor.shape)

Using max SMILES length: 100
Davis tensor shape: torch.Size([68, 100])
KIBA SMILES tensor shape: torch.Size([2111, 100])


c) Target Feature + BPE

In [ ]:
# Loading merge rules
merge_rules_path = os.path.join(root, "EDTA-main/data/davis/protein_codes_uniprot.txt")

merge_rules = []
with open(merge_rules_path, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split()
        if len(parts) == 2:
            merge_rules.append((parts[0], parts[1]))

print(f"Total merge rules: {len(merge_rules)}")

# Loading vocabulary
vocab_path = os.path.join(root, "EDTA-main/data/davis/subword_units_map_uniprot.csv")
vocab_df = pd.read_csv(vocab_path)
print(f"Total vocab size: {len(vocab_df)}")

token_to_id = dict(zip(vocab_df['index'], vocab_df['level_0']))
merge_rank = {(a, b): i for i, (a, b) in enumerate(merge_rules)}

def pad_target_feature(x, max_len):
    x = np.asarray(x)
    L, d = x.shape
    if L >= max_len:
        return x[:max_len]
    padded = np.zeros((max_len, d))
    padded[:L] = x
    return padded

# Fast BPE encoding
def apply_bpe_fast(sequence, merge_rank, token_to_id, max_len, unk_id=0):
    tokens = list(sequence)

    while True:
        best_rank = float("inf")
        best_idx  = -1
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            rank = merge_rank.get(pair, float("inf"))
            if rank < best_rank:
                best_rank = rank
                best_idx  = i

        if best_idx == -1:
            break

        merged = tokens[best_idx] + tokens[best_idx + 1]
        pair_a = tokens[best_idx]
        pair_b = tokens[best_idx + 1]
        new_tokens = []
        i = 0
        while i < len(tokens):
            if (i < len(tokens) - 1
                    and tokens[i] == pair_a
                    and tokens[i + 1] == pair_b):
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    ids = [token_to_id.get(t, unk_id) + 1 for t in tokens]

    if len(ids) >= max_len:
        return ids[:max_len]
    return ids + [0] * (max_len - len(ids))

def _load_one(args):
    file, folder_path, bpe_max_len, merge_rank, token_to_id = args
    protein_id = file.replace(".pkl", "")

    with open(os.path.join(folder_path, file), "rb") as f:
        data = pickle.load(f)

    seq_feat = pad_target_feature(data["seq_feat"], MAX_TARGET_LEN)
    bpe_ids  = apply_bpe_fast(str(data["seq"]), merge_rank, token_to_id, bpe_max_len)

    return protein_id, seq_feat, bpe_ids

MAX_TARGET_LEN    = 500
MAX_BPE_LEN_DAVIS = 400
MAX_BPE_LEN_KIBA  = 556

def load_target_features_fast(folder_path, bpe_max_len, max_workers=8):

    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".pkl")])
    protein_ids     = [None] * len(files)
    target_features = [None] * len(files)
    target_bpe      = [None] * len(files)

    args_list = [
        (file, folder_path, bpe_max_len, merge_rank, token_to_id)
        for file in files
    ]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_load_one, a): i for i, a in enumerate(args_list)}
        for future in as_completed(futures):
            i = futures[future]
            protein_ids[i], target_features[i], target_bpe[i] = future.result()

    target_features = torch.tensor(np.stack(target_features), dtype=torch.float32)
    target_bpe      = torch.tensor(np.array(target_bpe),      dtype=torch.long)

    return target_features, target_bpe, protein_ids

print("Loading DAVIS...")
davis_feat, davis_bpe, davis_ids = load_target_features_fast(
    os.path.join(root, "EDTA-main/data/davis/Target_Feature+BPE"),
    MAX_BPE_LEN_DAVIS
)
print(f"davis_feat : {davis_feat.shape}  dtype={davis_feat.dtype}")
print(f"davis_bpe  : {davis_bpe.shape}   dtype={davis_bpe.dtype}")

print("\nLoading KIBA...")
kiba_feat, kiba_bpe, kiba_ids = load_target_features_fast(
    os.path.join(root, "EDTA-main/data/kiba/Target_Feature+BPE"),
    MAX_BPE_LEN_KIBA
)
print(f"kiba_feat : {kiba_feat.shape}  dtype={kiba_feat.dtype}")
print(f"kiba_bpe  : {kiba_bpe.shape}   dtype={kiba_bpe.dtype}")

Total merge rules: 10120
Total vocab size: 16693
Loading DAVIS...
davis_feat : torch.Size([442, 500, 42])  dtype=torch.float32
davis_bpe  : torch.Size([442, 400])   dtype=torch.int64

Loading KIBA...
kiba_feat : torch.Size([228, 500, 42])  dtype=torch.float32
kiba_bpe  : torch.Size([228, 556])   dtype=torch.int64


d) Potential Residue

In [ ]:
davis_residue_path = os.path.join(root, "EDTA-main/data/davis/Potential_Residue_Feature")
kiba_residue_path  = os.path.join(root, "EDTA-main/data/kiba/Potential_Residue_Feature")

MAX_RESIDUE_LEN = 200
FEATURE_DIM = 67

def pad_residue_feature(mat):
    mat = np.asarray(mat)
    L, d = mat.shape

    # truncate
    if L >= MAX_RESIDUE_LEN:
        return mat[:MAX_RESIDUE_LEN]

    # pad
    padded = np.zeros((MAX_RESIDUE_LEN, d))
    padded[:L] = mat
    return padded

def load_residue_tensor(folder_path):
    residue_features = []
    protein_ids = []

    files = sorted(os.listdir(folder_path))
    for file in files:
        if not file.endswith(".pkl"):
            continue

        protein_id = file.replace(".pkl", "")
        protein_ids.append(protein_id)
        path = os.path.join(folder_path, file)
        with open(path, "rb") as f:
            data = pickle.load(f)

        residue_mat = data["seq_feat"]      # (L, 67)
        residue_mat = pad_residue_feature(residue_mat)
        residue_features.append(residue_mat)

    residue_tensor = torch.tensor(
        np.stack(residue_features),
        dtype=torch.float32
    )
    return residue_tensor, protein_ids

# LOAD DAVIS
print("Loading DAVIS residue features...")
davis_residue_tensor, davis_residue_ids = load_residue_tensor(davis_residue_path)
print("DAVIS residue tensor shape:", davis_residue_tensor.shape)

# LOAD KIBA
print("\nLoading KIBA residue features...")
kiba_residue_tensor, kiba_residue_ids = load_residue_tensor(kiba_residue_path)
print("KIBA residue tensor shape:", kiba_residue_tensor.shape)

Loading DAVIS residue features...
DAVIS residue tensor shape: torch.Size([442, 200, 67])

Loading KIBA residue features...
KIBA residue tensor shape: torch.Size([229, 200, 67])


Handling the missing Target_feature+BPE protein in KIBA

In [ ]:
kiba_bpe_set     = set(kiba_ids)
kiba_residue_set = set(kiba_residue_ids)
missing_from_bpe = kiba_residue_set - kiba_bpe_set
print(f"Proteins missing from BPE/Target_Feature: {missing_from_bpe}")
assert 'P78527' in missing_from_bpe

p78527_residue_row = kiba_residue_ids.index('P78527')
print(f"P78527 position in residue ID list: {p78527_residue_row}")

zero_target_feat = torch.zeros(
    1, kiba_feat.shape[1], kiba_feat.shape[2],
    dtype=kiba_feat.dtype
)

zero_target_bpe = torch.zeros(
    1, kiba_bpe.shape[1],
    dtype=kiba_bpe.dtype
)

# ---- Insert placeholder at the correct position ----
kiba_feat = torch.cat([
    kiba_feat[:p78527_residue_row],
    zero_target_feat,
    kiba_feat[p78527_residue_row:]
], dim=0)

kiba_bpe = torch.cat([
    kiba_bpe[:p78527_residue_row],
    zero_target_bpe,
    kiba_bpe[p78527_residue_row:]
], dim=0)

kiba_ids.insert(p78527_residue_row, 'P78527')

assert kiba_ids == kiba_residue_ids, \
    "KIBA protein ID lists still misaligned after fix"
assert kiba_feat.shape[0] == kiba_bpe.shape[0] == kiba_residue_tensor.shape[0] == len(kiba_ids), \
    "KIBA tensor row counts misaligned after fix"

print(f"KIBA proteins after fix : {len(kiba_ids)}")
print(f"kiba_feat shape         : {kiba_feat.shape}")
print(f"kiba_bpe shape          : {kiba_bpe.shape}")
print(f"kiba_residue shape      : {kiba_residue_tensor.shape}")
print(f"P78527 row in tensors   : {p78527_residue_row}  (zero-padded)")

assert davis_ids == davis_residue_ids, \
    f"DAVIS protein ID lists misaligned: {set(davis_ids) ^ set(davis_residue_ids)}"
print("DAVIS protein alignment : OK")

Proteins missing from BPE/Target_Feature: {'P78527'}
P78527 position in residue ID list: 128
KIBA proteins after fix : 229
kiba_feat shape         : torch.Size([229, 500, 42])
kiba_bpe shape          : torch.Size([229, 556])
kiba_residue shape      : torch.Size([229, 200, 67])
P78527 row in tensors   : 128  (zero-padded)
DAVIS protein alignment : OK


#### Pair expansion of collected features

In [ ]:
def make_index_map(keys):
    return {k: i for i, k in enumerate(keys)}

davis_drug_index   = make_index_map(davis_drug_keys)
davis_target_index = make_index_map(davis_ids)
kiba_drug_index    = make_index_map(kiba_drug_keys)
kiba_target_index  = make_index_map(kiba_ids)

#### PyTorch dataset construction

In [ ]:
import ast
import torch
from torch.utils.data import Dataset, DataLoader, Subset

# LOAD OFFICIAL SPLIT FILES
def load_fold_indices(path):
    with open(path, "r") as f:
        content = f.read().strip()
    return ast.literal_eval(content)

# DAVIS
davis_test_indices = load_fold_indices(os.path.join(root, "EDTA-main", "data", "davis", "folds", "test_fold_setting1.txt"))
davis_train_folds = load_fold_indices(os.path.join(root, "EDTA-main", "data", "davis", "folds", "train_fold_setting1.txt"))

# KIBA
kiba_test_indices = load_fold_indices(os.path.join(root, "EDTA-main", "data", "kiba", "folds", "test_fold_setting1.txt"))
kiba_train_folds = load_fold_indices(os.path.join(root, "EDTA-main", "data", "kiba", "folds", "train_fold_setting1.txt"))

print(f"DAVIS test set size      : {len(davis_test_indices)}")
print(f"DAVIS folds (5)          : {[len(f) for f in davis_train_folds]}")
print(f"KIBA  test set size      : {len(kiba_test_indices)}")
print(f"KIBA  folds (5)          : {[len(f) for f in kiba_train_folds]}")

DAVIS test set size      : 5010
DAVIS folds (5)          : [5010, 5009, 5009, 5009, 5009]
KIBA  test set size      : 19709
KIBA  folds (5)          : [19709, 19709, 19709, 19709, 19709]


In [ ]:
class DTIDataset(Dataset):
    """
    Stores only index arrays. Slices base feature tensors on demand
    in __getitem__ — no duplication of feature matrices in memory.
    """
    def __init__(
        self,
        d_indices, t_indices, labels,
        smiles_tensor,
        drug_atomic_tensor,
        target_feat_tensor,
        target_bpe_tensor,
        residue_tensor
    ):
        assert len(d_indices) == len(t_indices) == len(labels)
        self.d_indices          = d_indices
        self.t_indices          = t_indices
        self.labels             = labels
        self.smiles_tensor      = smiles_tensor
        self.drug_atomic_tensor = drug_atomic_tensor
        self.target_feat_tensor = target_feat_tensor
        self.target_bpe_tensor  = target_bpe_tensor
        self.residue_tensor     = residue_tensor

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        d_i = self.d_indices[idx]
        t_i = self.t_indices[idx]
        return {
            "smiles"      : self.smiles_tensor[d_i],
            "drug_atomic" : self.drug_atomic_tensor[d_i],
            "target_feat" : self.target_feat_tensor[t_i],
            "target_bpe"  : self.target_bpe_tensor[t_i],
            "residue"     : self.residue_tensor[t_i],
            "label"       : self.labels[idx]
        }

def build_pair_indices(df, drug_map, target_map):
    df = df.copy()
    df["d_idx"] = df["ligand"].map(drug_map)
    df["t_idx"] = df["protein"].map(target_map)

    unmatched_drugs = df[df["d_idx"].isna()]["ligand"].unique()
    if len(unmatched_drugs) > 0:
        print(f"WARNING: {len(unmatched_drugs)} unmatched drug SMILES:")
        for s in unmatched_drugs[:5]:
            print(f"  '{s}'")
    assert len(unmatched_drugs) == 0, \
        f"{len(unmatched_drugs)} drug SMILES in CSV not found in drug_map"

    unmatched_targets = df[df["t_idx"].isna()]["protein"].unique()
    if len(unmatched_targets) > 0:
        print(f"WARNING: {len(unmatched_targets)} unmatched protein IDs:")
        for s in unmatched_targets[:5]:
            print(f"  '{s}'")
    assert len(unmatched_targets) == 0, \
        f"{len(unmatched_targets)} protein IDs in CSV not found in target_map"

    assert df["d_idx"].isna().sum() == 0
    assert df["t_idx"].isna().sum() == 0

    df[["d_idx", "t_idx"]] = df[["d_idx", "t_idx"]].astype(int)
    d_indices = torch.from_numpy(df["d_idx"].values.astype("int64"))
    t_indices = torch.from_numpy(df["t_idx"].values.astype("int64"))
    labels    = torch.tensor(df["label"].values, dtype=torch.float32)

    print(f"Total pairs built: {len(labels)}")
    return d_indices, t_indices, labels

davis_d_idx, davis_t_idx, davis_labels = build_pair_indices(davis_data, davis_drug_index, davis_target_index)
kiba_d_idx, kiba_t_idx, kiba_labels = build_pair_indices(kiba_data, kiba_drug_index, kiba_target_index)

davis_dataset = DTIDataset(
    davis_d_idx, davis_t_idx, davis_labels,
    davis_smiles_tensor, davis_drug_tensor,
    davis_feat, davis_bpe, davis_residue_tensor
)

kiba_dataset = DTIDataset(
    kiba_d_idx, kiba_t_idx, kiba_labels,
    kiba_smiles_tensor, kiba_drug_tensor,
    kiba_feat, kiba_bpe, kiba_residue_tensor
)

print(f"\nDAVIS full dataset size : {len(davis_dataset)}")
print(f"KIBA  full dataset size : {len(kiba_dataset)}")

Total pairs built: 30056
Total pairs built: 118254

DAVIS full dataset size : 30056
KIBA  full dataset size : 118254


#### Sample level setting

In [ ]:
def make_official_splits(dataset, train_folds, test_indices):
    """
    Uses the official fold index files to produce:
      - test_set     : Subset using the fixed test indices
      - fold_splits  : List of 5 (train_subset, val_subset) tuples
                       following the 4-train / 1-val rotation
    All subsets reference the same base dataset — no data is copied.
    """
    test_set = Subset(dataset, test_indices)

    fold_splits = []
    for val_fold_idx in range(5):
        val_indices   = train_folds[val_fold_idx]
        train_indices = []
        for i, fold in enumerate(train_folds):
            if i != val_fold_idx:
                train_indices.extend(fold)

        train_subset = Subset(dataset, train_indices)
        val_subset   = Subset(dataset, val_indices)
        fold_splits.append((train_subset, val_subset))

    return test_set, fold_splits

davis_test_set, davis_fold_splits = make_official_splits(davis_dataset, davis_train_folds, davis_test_indices)
kiba_test_set,  kiba_fold_splits  = make_official_splits(kiba_dataset,  kiba_train_folds,  kiba_test_indices)

print("DAVIS splits:")
print(f"  Test : {len(davis_test_set)}")
for i, (tr, vl) in enumerate(davis_fold_splits):
    print(f"  Fold {i+1} → train={len(tr)}  val={len(vl)}")

print("\nKIBA splits:")
print(f"  Test : {len(kiba_test_set)}")
for i, (tr, vl) in enumerate(kiba_fold_splits):
    print(f"  Fold {i+1} → train={len(tr)}  val={len(vl)}")

DAVIS splits:
  Test : 5010
  Fold 1 → train=20036  val=5010
  Fold 2 → train=20037  val=5009
  Fold 3 → train=20037  val=5009
  Fold 4 → train=20037  val=5009
  Fold 5 → train=20037  val=5009

KIBA splits:
  Test : 19709
  Fold 1 → train=78836  val=19709
  Fold 2 → train=78836  val=19709
  Fold 3 → train=78836  val=19709
  Fold 4 → train=78836  val=19709
  Fold 5 → train=78836  val=19709


Dataloaders

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = 2

PIN_MEMORY = torch.cuda.is_available()

def make_loaders(train_subset, val_subset, test_subset, num_workers=0):
    train_loader = DataLoader(
        train_subset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=num_workers, pin_memory=PIN_MEMORY,
        persistent_workers=(num_workers > 0)
    )
    val_loader = DataLoader(
        val_subset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=num_workers, pin_memory=PIN_MEMORY,
        persistent_workers=(num_workers > 0)
    )
    test_loader = DataLoader(
        test_subset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=num_workers, pin_memory=PIN_MEMORY,
        persistent_workers=(num_workers > 0)
    )
    return train_loader, val_loader, test_loader

davis_loaders = [make_loaders(tr, vl, davis_test_set, num_workers=2) for tr, vl in davis_fold_splits]
kiba_loaders  = [make_loaders(tr, vl, kiba_test_set, num_workers=0)  for tr, vl in kiba_fold_splits]

print("DataLoaders ready for 5-fold cross-validation")

def check_batch(loader, name):
    batch = next(iter(loader))
    print(f"\n{name} sample batch:")
    for k, v in batch.items():
        print(f"  {k:15s} : {v.shape}  dtype={v.dtype}")

davis_train_loader_0, davis_val_loader_0, davis_test_loader = davis_loaders[0]
kiba_train_loader_0,  kiba_val_loader_0,  kiba_test_loader  = kiba_loaders[0]

check_batch(davis_train_loader_0, "DAVIS fold-0 train")
check_batch(davis_val_loader_0,   "DAVIS fold-0 val")
check_batch(davis_test_loader,    "DAVIS test")
check_batch(kiba_train_loader_0,  "KIBA  fold-0 train")

DataLoaders ready for 5-fold cross-validation

DAVIS fold-0 train sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 24])  dtype=torch.float32
  target_feat     : torch.Size([32, 500, 42])  dtype=torch.float32
  target_bpe      : torch.Size([32, 400])  dtype=torch.int64
  residue         : torch.Size([32, 200, 67])  dtype=torch.float32
  label           : torch.Size([32])  dtype=torch.float32

DAVIS fold-0 val sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 24])  dtype=torch.float32
  target_feat     : torch.Size([32, 500, 42])  dtype=torch.float32
  target_bpe      : torch.Size([32, 400])  dtype=torch.int64
  residue         : torch.Size([32, 200, 67])  dtype=torch.float32
  label           : torch.Size([32])  dtype=torch.float32

DAVIS test sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 2

In [ ]:
import torch.nn as nn

EMBED_DIM = 128
SMILES_VOCAB_SIZE = len(char_to_int) + 1
BPE_VOCAB_SIZE    = len(vocab_df) + 1

DRUG_ATOMIC_DIM  = 24
TARGET_FEAT_DIM  = 42
RESIDUE_DIM      = 67


class EmbeddingModule(nn.Module):
    def __init__(
        self,
        smiles_vocab_size: int = SMILES_VOCAB_SIZE,
        bpe_vocab_size:    int = BPE_VOCAB_SIZE,
        drug_atomic_dim:   int = DRUG_ATOMIC_DIM,
        target_feat_dim:   int = TARGET_FEAT_DIM,
        residue_dim:       int = RESIDUE_DIM,
        embed_dim:         int = EMBED_DIM,
    ):
        super().__init__()

        self.smiles_embed      = nn.Embedding(smiles_vocab_size, embed_dim, padding_idx=0)
        self.target_bpe_embed  = nn.Embedding(bpe_vocab_size,    embed_dim, padding_idx=0)
        self.drug_atomic_proj  = nn.Linear(drug_atomic_dim, embed_dim, bias=False)
        self.target_feat_proj  = nn.Linear(target_feat_dim, embed_dim, bias=False)
        self.residue_proj      = nn.Linear(residue_dim,     embed_dim, bias=False)

    def forward(self, smiles, drug_atomic, target_feat, target_bpe, residue):
        smiles_emb      = self.smiles_embed(smiles)
        target_bpe_emb  = self.target_bpe_embed(target_bpe)
        drug_atomic_emb = self.drug_atomic_proj(drug_atomic)
        target_feat_emb = self.target_feat_proj(target_feat)
        residue_emb     = self.residue_proj(residue)

        return smiles_emb, drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb

In [ ]:
class Conv1dBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )
        self.bn   = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class CNN1DBlock(nn.Module):
    CHANNELS     = [32, 64, 128, 256]
    KERNELS_SMALL = [4,  6,   6,   4]
    KERNELS_LARGE = [4,  8,  12,   8]

    def __init__(self, embed_dim: int = EMBED_DIM, kernel_preset: str = 'small'):
        super().__init__()

        assert kernel_preset in ('small', 'large'), \
            "kernel_preset must be 'small' [4,6,6,4] or 'large' [4,8,12,8]"

        kernels = self.KERNELS_SMALL if kernel_preset == 'small' else self.KERNELS_LARGE
        in_channels_list = [embed_dim] + self.CHANNELS[:-1]

        self.layers = nn.ModuleList([
            Conv1dBlock(in_ch, out_ch, ks)
            for in_ch, out_ch, ks in zip(in_channels_list, self.CHANNELS, kernels)
        ])

    def forward(self, x):
        x = x.transpose(1, 2)
        for layer in self.layers:
            x = layer(x)

        x = x.max(dim=2).values
        return x

class AllCNNBlocks(nn.Module):
    def __init__(self, embed_dim: int = EMBED_DIM):
        super().__init__()
        self.drug_atomic_cnn = CNN1DBlock(embed_dim, kernel_preset='small')
        self.target_feat_cnn = CNN1DBlock(embed_dim, kernel_preset='large')
        self.target_bpe_cnn  = CNN1DBlock(embed_dim, kernel_preset='small')
        self.residue_cnn     = CNN1DBlock(embed_dim, kernel_preset='large')

    def forward(self, drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb):
        drug_atomic_vec = self.drug_atomic_cnn(drug_atomic_emb)
        target_feat_vec = self.target_feat_cnn(target_feat_emb)
        target_bpe_vec  = self.target_bpe_cnn(target_bpe_emb)
        residue_vec     = self.residue_cnn(residue_emb)

        return drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec

In [ ]:
class BiLSTMWithAttention(nn.Module):
    def __init__(
        self,
        embed_dim:   int = EMBED_DIM,
        hidden_size: int = 128,
        num_layers:  int = 2,
        dropout:     float = 0.1,
    ):
        super().__init__()

        self.hidden_size = hidden_size

        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.attn_score = nn.Linear(hidden_size * 2, 1, bias=True)

    def forward(self, smiles_emb, smiles_tok):
        lstm_out, _ = self.bilstm(smiles_emb)
        pad_mask = (smiles_tok != 0)

        scores = self.attn_score(lstm_out).squeeze(-1)
        scores = scores.masked_fill(~pad_mask, float('-inf'))
        attn_weights = torch.softmax(scores, dim=1)

        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

        context = torch.bmm(
            attn_weights.unsqueeze(1),
            lstm_out
        ).squeeze(1)

        return context

class ConcatenationLayer(nn.Module):

    def forward(self, smiles_vec, drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec,):
        return torch.cat(
            [smiles_vec, drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec],
            dim=1)

In [ ]:
class SelfAttentionLayer(nn.Module):

    def __init__(self, input_dim: int = 1280, attn_dim: int = 128):
        super().__init__()

        self.scale    = attn_dim ** -0.5
        self.W_q = nn.Linear(input_dim, attn_dim, bias=True)
        self.W_k = nn.Linear(input_dim, attn_dim, bias=True)
        self.W_v = nn.Linear(input_dim, attn_dim, bias=True)
        self.W_o = nn.Linear(attn_dim,  input_dim, bias=True)

        self.layer_norm = nn.LayerNorm(input_dim)
        self.dropout    = nn.Dropout(p=0.1)

    def forward(self, x):
        x_seq = x.unsqueeze(1)
        Q = self.W_q(x_seq)
        K = self.W_k(x_seq)
        V = self.W_v(x_seq)

        attn_scores  = torch.bmm(Q, K.transpose(1, 2)) * self.scale
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attended = torch.bmm(attn_weights, V)
        attended = self.W_o(attended)

        out = self.layer_norm(x_seq + attended)
        return out.squeeze(1)    

In [ ]:
class FCNNOutput(nn.Module):
    def __init__(self, input_dim: int = 1280, dropout: float = 0.1):
        super().__init__()

        dims = [input_dim, 512, 256, 128, 64]

        fc_layers = []
        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            fc_layers.extend([
                nn.Linear(in_dim, out_dim, bias=False),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout),
            ])

        self.fc_block     = nn.Sequential(*fc_layers)
        self.output_layer = nn.Linear(64, 1, bias=True)

    def forward(self, x):
        x = self.fc_block(x)
        x = self.output_layer(x)
        return x.squeeze(-1)

class EDTA(nn.Module):
    def __init__(
        self,
        smiles_vocab_size: int   = SMILES_VOCAB_SIZE,
        bpe_vocab_size:    int   = BPE_VOCAB_SIZE,
        embed_dim:         int   = EMBED_DIM,
        lstm_hidden:       int   = 128,
        lstm_layers:       int   = 2,
        lstm_dropout:      float = 0.1,
        attn_dim:          int   = 128,
        fcnn_dropout:      float = 0.1,
    ):
        super().__init__()

        self.embedding   = EmbeddingModule(
            smiles_vocab_size = smiles_vocab_size,
            bpe_vocab_size    = bpe_vocab_size,
            embed_dim         = embed_dim,
        )
        self.cnn_blocks  = AllCNNBlocks(embed_dim=embed_dim)
        self.bilstm_attn = BiLSTMWithAttention(
            embed_dim   = embed_dim,
            hidden_size = lstm_hidden,
            num_layers  = lstm_layers,
            dropout     = lstm_dropout,
        )
        self.concat      = ConcatenationLayer()
        self.self_attn   = SelfAttentionLayer(input_dim=1280, attn_dim=attn_dim)
        self.fcnn        = FCNNOutput(input_dim=1280, dropout=fcnn_dropout)

    def forward(self, smiles, drug_atomic, target_feat, target_bpe, residue):

        smiles_emb, drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb = \
            self.embedding(smiles, drug_atomic, target_feat, target_bpe, residue)

        drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec = \
            self.cnn_blocks(drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb)

        smiles_vec = self.bilstm_attn(smiles_emb, smiles)

        fused    = self.concat(smiles_vec, drug_atomic_vec,
                               target_feat_vec, target_bpe_vec, residue_vec)

        attended = self.self_attn(fused)

        return self.fcnn(attended)

model = EDTA()

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from scipy import stats

import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP_wrap
from torch.utils.data.distributed import DistributedSampler

def ddp_setup():
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))


def ddp_cleanup():
    dist.destroy_process_group()


def is_ddp():
    return "RANK" in os.environ


def is_main_process():
    if not is_ddp():
        return True
    return dist.get_rank() == 0


def get_device():
    if is_ddp():
        # LOCAL_RANK tells this process which GPU it owns
        local_rank = int(os.environ["LOCAL_RANK"])
        return torch.device(f"cuda:{local_rank}")
    else:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
PIN_MEMORY = torch.cuda.is_available()


def make_loaders(train_subset, val_subset, test_subset, num_workers=2):
    if is_ddp():
        train_sampler = DistributedSampler(train_subset, shuffle=True)
        train_loader = DataLoader(
            train_subset,
            batch_size         = BATCH_SIZE,
            shuffle            = False,         # sampler handles this
            sampler            = train_sampler,
            num_workers        = num_workers,
            pin_memory         = PIN_MEMORY,
            persistent_workers = (num_workers > 0),
            drop_last          = True,
        )
    else:
        train_sampler = None
        train_loader = DataLoader(
            train_subset,
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = num_workers,
            pin_memory         = PIN_MEMORY,
            persistent_workers = (num_workers > 0),
        )

    val_loader = DataLoader(
        val_subset,
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = num_workers,
        pin_memory         = PIN_MEMORY,
        persistent_workers = (num_workers > 0),
    )
    test_loader = DataLoader(
        test_subset,
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = num_workers,
        pin_memory         = PIN_MEMORY,
        persistent_workers = (num_workers > 0),
    )

    return train_loader, val_loader, test_loader, train_sampler

def batch_to_device(batch, device):
    return {
        k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

def metrics_mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


def metrics_ci(y_true, y_pred, chunk=3000):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    n = len(y_true)

    n_total    = 0
    concordant = 0.0

    for start in range(0, n, chunk):
        end  = min(start + chunk, n)
        rows = slice(start, end)

        dt = y_true[rows, None] - y_true[None, :]
        dp = y_pred[rows, None] - y_pred[None, :]

        row_idx = np.arange(start, end)
        col_idx = np.arange(n)
        upper   = row_idx[:, None] < col_idx[None, :]
        mask    = upper & (dt != 0)

        n_total    += int(mask.sum())
        concordant += float(np.sum((dt[mask] * dp[mask]) > 0))
        concordant += 0.5 * float(np.sum(dp[mask] == 0))

    return concordant / n_total if n_total > 0 else 0.0


def metrics_pearson(y_true, y_pred):
    r, _ = stats.pearsonr(y_true, y_pred)
    return float(r)


def metrics_r2m(y_true, y_pred):
    ss_res  = np.sum((y_true - y_pred) ** 2)
    ss_tot  = np.sum((y_true - np.mean(y_true)) ** 2)
    r2      = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

    ss_pred0 = np.sum(y_pred ** 2)
    ss_cross = np.sum(y_true * y_pred)
    k        = ss_cross / ss_pred0 if ss_pred0 > 0 else 1.0
    ss_res0  = np.sum((y_true - k * y_pred) ** 2)
    r02      = 1.0 - ss_res0 / ss_tot if ss_tot > 0 else 0.0

    r2m = r2 * (1.0 - np.sqrt(abs(r2 - r02)))
    return float(r2m)

def evaluate_fast(model, loader, device, criterion):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n_batches = 0.0, 0

    with torch.no_grad():
        for batch in loader:
            batch = batch_to_device(batch, device)
            with autocast("cuda"):
                preds = model(
                    batch['smiles'],
                    batch['drug_atomic'],
                    batch['target_feat'],
                    batch['target_bpe'],
                    batch['residue'],
                )
            loss = criterion(preds.float(), batch['label'])
            total_loss += loss.item()
            n_batches  += 1
            all_preds.append(preds.float().cpu().numpy())
            all_labels.append(batch['label'].cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    return total_loss / n_batches, metrics_mse(y_true, y_pred)

def evaluate_full(model, loader, device, criterion=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n_batches = 0.0, 0

    with torch.no_grad():
        for batch in loader:
            batch = batch_to_device(batch, device)
            with autocast("cuda"):
                preds = model(
                    batch['smiles'],
                    batch['drug_atomic'],
                    batch['target_feat'],
                    batch['target_bpe'],
                    batch['residue'],
                )
            if criterion is not None:
                loss = criterion(preds.float(), batch['label'])
                total_loss += loss.item()
                n_batches  += 1
            all_preds.append(preds.float().cpu().numpy())
            all_labels.append(batch['label'].cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    result = {
        'mse':     metrics_mse(y_true, y_pred),
        'ci':      metrics_ci(y_true, y_pred),
        'pearson': metrics_pearson(y_true, y_pred),
        'r2m':     metrics_r2m(y_true, y_pred),
    }
    if criterion is not None:
        result['loss'] = total_loss / n_batches
    return result

def train_one_fold(
    fold_idx,
    train_loader,
    val_loader,
    test_loader,
    train_sampler,
    device,
    dataset_name,
    num_epochs   = 200,
    save_dir     = "/kaggle/working/checkpoints",
    smiles_vocab = SMILES_VOCAB_SIZE,
    bpe_vocab    = BPE_VOCAB_SIZE,
):
    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(
        save_dir, f"{dataset_name}_fold{fold_idx+1}_best.pt"
    )

    model = EDTA(
        smiles_vocab_size = smiles_vocab,
        bpe_vocab_size    = bpe_vocab,
    ).to(device)

    if hasattr(torch, 'compile'):
        model = torch.compile(model)

    if is_ddp():

        model = DDP_wrap(model, device_ids=[device.index])
        raw_model = model.module
    else:
        raw_model = model

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)

    scaler = GradScaler()

    best_val_loss    = float('inf')
    best_val_metrics = {}
    train_losses     = []
    val_losses       = []

    if is_main_process():
        print(f"\n{'='*60}")
        print(f"  {dataset_name.upper()}  |  Fold {fold_idx+1}/5")
        print(f"{'='*60}")
        print(f"  {'Epoch':>6}  {'Train MSE':>10}  {'Val MSE':>10}  "
              f"{'Val CI':>8}  {'Val r2m':>8}")
        print(f"  {'-'*6}  {'-'*10}  {'-'*10}  {'-'*8}  {'-'*8}")

    for epoch in range(1, num_epochs + 1):

        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        model.train()
        epoch_loss = 0.0
        n_batches  = 0

        for batch in train_loader:
            batch = batch_to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                preds = model(
                    batch['smiles'],
                    batch['drug_atomic'],
                    batch['target_feat'],
                    batch['target_bpe'],
                    batch['residue'],
                )
                loss = criterion(preds, batch['label'])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            n_batches  += 1

        train_mse = epoch_loss / n_batches
        train_losses.append(train_mse)
        if is_main_process():
            val_loss, val_mse = evaluate_fast(model, val_loader, device, criterion)
            val_losses.append(val_mse)

            if val_loss < best_val_loss:
                best_val_loss = val_loss


                full_val = evaluate_full(model, val_loader, device, criterion)
                best_val_metrics = {**full_val, 'epoch': epoch}
                torch.save(raw_model.state_dict(), ckpt_path)

            if epoch % 10 == 0 or epoch == 1:
                ci_disp  = f"{best_val_metrics.get('ci',  0):.4f}"
                r2m_disp = f"{best_val_metrics.get('r2m', 0):.4f}"
                marker   = " ◀ best" if val_loss == best_val_loss else ""
                print(
                    f"  {epoch:>6}  {train_mse:>10.4f}  "
                    f"{val_mse:>10.4f}  {ci_disp:>8}  "
                    f"{r2m_disp:>8}{marker}"
                )
        else:
            val_losses.append(0.0)

    test_metrics = {}
    if is_main_process():
        state = torch.load(ckpt_path, map_location=device)
        raw_model.load_state_dict(state)
        test_metrics = evaluate_full(raw_model, test_loader, device)

        print(f"\n  Best val epoch : {best_val_metrics['epoch']}")
        print(f"  Val  → MSE={best_val_metrics['mse']:.4f}  "
              f"CI={best_val_metrics['ci']:.4f}  "
              f"r={best_val_metrics['pearson']:.4f}  "
              f"r2m={best_val_metrics['r2m']:.4f}")
        print(f"  Test → MSE={test_metrics['mse']:.4f}  "
              f"CI={test_metrics['ci']:.4f}  "
              f"r={test_metrics['pearson']:.4f}  "
              f"r2m={test_metrics['r2m']:.4f}")

    return best_val_metrics, test_metrics, train_losses, val_losses

def run_cross_validation(
    dataset_name,
    fold_loaders,
    device,
    num_epochs = 200,
    save_dir   = "/kaggle/working/checkpoints",
):

    all_val_metrics  = []
    all_test_metrics = []
    all_train_losses = []
    all_val_losses   = []

    for fold_idx, (train_loader, val_loader, test_loader, train_sampler) in enumerate(fold_loaders):
        val_m, test_m, tr_loss, vl_loss = train_one_fold(
            fold_idx      = fold_idx,
            train_loader  = train_loader,
            val_loader    = val_loader,
            test_loader   = test_loader,
            train_sampler = train_sampler,
            device        = device,
            dataset_name  = dataset_name,
            num_epochs    = num_epochs,
            save_dir      = save_dir,
        )
        all_val_metrics.append(val_m)
        all_test_metrics.append(test_m)
        all_train_losses.append(tr_loss)
        all_val_losses.append(vl_loss)

    if is_main_process():
        print(f"\n{'='*60}")
        print(f"  {dataset_name.upper()}  5-FOLD CROSS-VALIDATION SUMMARY")
        print(f"{'='*60}")
        print(f"  {'':8}  {'MSE':>8}  {'CI':>8}  {'Pearson':>8}  {'r2m':>8}")
        print(f"  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")

        for i, m in enumerate(all_test_metrics):
            print(
                f"  Fold {i+1:>3}  "
                f"{m['mse']:>8.4f}  {m['ci']:>8.4f}  "
                f"{m['pearson']:>8.4f}  {m['r2m']:>8.4f}"
            )

        metrics_keys = ['mse', 'ci', 'pearson', 'r2m']
        avgs = {k: np.mean([m[k] for m in all_test_metrics]) for k in metrics_keys}
        stds = {k: np.std( [m[k] for m in all_test_metrics]) for k in metrics_keys}

        print(
            f"\n  {'Average':8}  "
            f"{avgs['mse']:>8.4f}  {avgs['ci']:>8.4f}  "
            f"{avgs['pearson']:>8.4f}  {avgs['r2m']:>8.4f}"
        )
        print(
            f"  {'Std Dev':8}  "
            f"{stds['mse']:>8.4f}  {stds['ci']:>8.4f}  "
            f"{stds['pearson']:>8.4f}  {stds['r2m']:>8.4f}"
        )

    return {
        'val_metrics':   all_val_metrics,
        'test_metrics':  all_test_metrics,
        'train_losses':  all_train_losses,
        'val_losses':    all_val_losses,
    }

if __name__ == "__main__":
    if is_ddp():
        ddp_setup()
    device = get_device()

    if is_main_process():
        print(f"Device : {device}")
        if device.type == 'cuda':
            print(f"GPU    : {torch.cuda.get_device_name(device)}")
        if is_ddp():
            print(f"DDP    : {dist.get_world_size()} processes")

    davis_fold_loaders = [
        make_loaders(tr, vl, davis_test_set, num_workers=2)
        for tr, vl in davis_fold_splits
    ]

    kiba_fold_loaders = [
        make_loaders(tr, vl, kiba_test_set, num_workers=0)
        for tr, vl in kiba_fold_splits
    ]

    davis_results = run_cross_validation(
        dataset_name = 'davis',
        fold_loaders = davis_fold_loaders,
        device       = device,
        num_epochs   = 200,
        save_dir     = "/kaggle/working/checkpoints",
    )

    kiba_results = run_cross_validation(
        dataset_name = 'kiba',
        fold_loaders = kiba_fold_loaders,
        device       = device,
        num_epochs   = 200,
        save_dir     = "/kaggle/working/checkpoints",
    )

    if is_ddp():
        ddp_cleanup()

Device : cuda
GPU    : Tesla T4

  DAVIS  |  Fold 1/5
   Epoch   Train MSE     Val MSE    Val CI   Val r2m
  ------  ----------  ----------  --------  --------


W0414 15:56:47.683000 55 torch/_inductor/utils.py:1679] [0/0_1] Not enough SMs to use max_autotune_gemm mode


       1     14.3975      2.7854    0.6870    1.5195 ◀ best
      10      0.5524      0.4396    0.8195    0.3756 ◀ best
      20      0.4444      0.3755    0.8340    0.4838
      30      0.3991      0.3518    0.8427    0.5018
      40      0.3502      0.3138    0.8495    0.5527
      50      0.3195      0.2822    0.8572    0.6401 ◀ best
      60      0.3018      0.2889    0.8608    0.6194
      70      0.2702      0.2746    0.8608    0.6194
      80      0.2636      0.2979    0.8653    0.6353
      90      0.2475      0.2669    0.8653    0.6353
     100      0.2302      0.2610    0.8659    0.6548
     110      0.2173      0.2520    0.8655    0.6457 ◀ best
     120      0.1974      0.2524    0.8705    0.6759
     130      0.1955      0.2418    0.8718    0.6716
     140      0.1791      0.2341    0.8772    0.7039 ◀ best
     150      0.1733      0.2481    0.8772    0.7039
     160      0.1642      0.2457    0.8772    0.7039
     170      0.1549      0.2291    0.8756    0.6757
     180   

KeyboardInterrupt: 